# Inspect base Jacobian stats

Load the per-sublayer h5 files under `onnx_models/smoll/jacobian_stats/`
(produced by `scripts/compute_base_jacobian_stats.py`) and look at the
per-(sample, position) smallest singular value and condition number for each
sublayer.

In [ ]:
import glob
import os

import h5py
import numpy as np
import matplotlib.pyplot as plt

STATS_DIR = "onnx_models/smoll/jacobian_stats"

In [ ]:
layer_paths = sorted(glob.glob(os.path.join(STATS_DIR, "*.h5")))
layer_names = [os.path.splitext(os.path.basename(p))[0] for p in layer_paths]
print("layers found:", layer_names)

## Load stats for every layer

Each layer's file has its own `(n_samples, seq)` datasets for smallest
singular value and condition number, plus `base_seqs` and
`n_samples`/`seq`/`d_model` attrs.

In [ ]:
def load_layer_stats(path: str) -> dict[str, np.ndarray]:
    with h5py.File(path, "r") as h5f:
        return {
            "smallest_sv": h5f["smallest_singular_value"][:],
            "condition_number": h5f["condition_number"][:],
            "base_seqs": h5f["base_seqs"][:],
            "attrs": dict(h5f.attrs),
        }

stats = {name: load_layer_stats(path) for name, path in zip(layer_names, layer_paths)}
for name, res in stats.items():
    sv, cond = res["smallest_sv"], res["condition_number"]
    print(f"{name:>20s}  smallest_sv shape={sv.shape}  attrs={res['attrs']}  "
          f"range=[{sv.min():.4g}, {sv.max():.4g}]  "
          f"condition_number range=[{cond.min():.4g}, {cond.max():.4g}]"
          if sv.size else f"{name:>20s}  (no chunks written yet)")

## A few raw samples for one layer

Note: if a run is still in progress, a layer's datasets may only have as
many rows as have been flushed so far -- that's expected with the chunked
writer.

In [ ]:
layer_name = layer_names[0]
n_show = 5

print(layer_name, "rows written:", stats[layer_name]["smallest_sv"].shape[0])
print("smallest_sv:\n", stats[layer_name]["smallest_sv"][:n_show])
print("condition_number:\n", stats[layer_name]["condition_number"][:n_show])

## Distribution of smallest singular value per layer

Flattened over samples and sequence positions (whatever has been written so
far for each layer).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, res in stats.items():
    sv = res["smallest_sv"]
    if sv.size == 0:
        continue
    ax.hist(sv.ravel(), bins=50, histtype="step", label=name)
ax.set_xlabel("smallest singular value")
ax.set_ylabel("count")
ax.set_title("Smallest singular value by layer")
ax.legend(fontsize=8)
plt.show()

## Distribution of condition number per layer (log scale)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, res in stats.items():
    cond = res["condition_number"]
    if cond.size == 0:
        continue
    ax.hist(np.log10(cond.ravel()), bins=50, histtype="step", label=name)
ax.set_xlabel("log10(condition number)")
ax.set_ylabel("count")
ax.set_title("Condition number by layer")
ax.legend(fontsize=8)
plt.show()

## Mean stats per sequence position, per layer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, res in stats.items():
    sv, cond = res["smallest_sv"], res["condition_number"]
    if sv.size == 0:
        continue
    axes[0].plot(sv.mean(axis=0), marker="o", label=name)
    axes[1].plot(cond.mean(axis=0), marker="o", label=name)

axes[0].set_title("Mean smallest singular value vs. position")
axes[0].set_xlabel("sequence position")
axes[1].set_title("Mean condition number vs. position")
axes[1].set_xlabel("sequence position")
axes[1].set_yscale("log")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()